# Bayesian Inference: CIP Dose-Response Model

This notebook runs all MCMC fits and computes the intermediate results (NetCDF posteriors, CSV summary tables) that the figure notebooks consume.  **Run this notebook before `main_figures.ipynb` or `supp_figures.ipynb`.**

## What this notebook does

1. **Model fits — standard condition** (Figs. 1, S2–S5): fits the five-parameter CIP logistic model simultaneously to all 11 CIP concentrations for each of the five biological replicates in M9 + 0.2% glucose.
2. **Independent logistic fits** (Figs. 1B, S2–S5 panel B): fits a plain logistic to each concentration separately, providing the comparison baseline.
3. **r(c) and K(c) extraction** (Figs. 1C–D, S2–S5): propagates posterior samples through the analytical expressions to obtain concentration-dependent growth-rate and carrying-capacity curves.
4. **MIC with fewer concentrations** (Fig. 2A): refits the model using only the *n* smallest CIP concentrations (n = 3, ..., 11) to test how many concentrations are needed for a reliable MIC estimate.
5. **MIC robustness / CV analysis** (Fig. 2B): computes the coefficient of variation of the MIC and each individual parameter across the five replicates.
6. **New medium conditions** (Figs. 3–4): fits the model to the twelve medium-perturbation conditions and assembles the parameter table.
7. **Fixed-α fits** (Fig. S6): repeats step 6 with α pinned to its global mean, for comparison.

## Output files

| File | Used in |
|---|---|
| `results/cip_model_standard_{rep}_12h.nc` | Figs. 1, S2–S5, 2B |
| `results/cip_logistic_standard_{rep}_dose_{c}_12h.nc` | Figs. 1B, S2–S5 |
| `results/rK_{rep}_12h.csv` | Figs. 1C–D, S2–S5 |
| `results/mic_calculations_add_{n}_12h.nc` | Fig. 2A |
| `results/mic_convergence_data_linear_add.csv` | Fig. 2A |
| `results/cip_model_cv.csv` | Fig. 2B |
| `results/cip_model_{medium}_{cond}_12h.nc` | Figs. 3–4 |
| `results/parameters_table.csv` | Figs. 3–4, Table 1 |
| `results/cip_model_{medium}_{cond}_12h_fixed_alpha.nc` | Fig. S6 |
| `results/parameters_table_fixed_alpha.csv` | Fig. S6 |

In [ ]:
from functions import *

---
## 1. Model fits — standard condition
*(Posteriors used in Figs. 1, S2–S5)*

We fit the five-parameter CIP logistic model (manuscript Eq. 5) **simultaneously** to all 11 CIP concentrations for each biological replicate.
All curves are truncated at 12 h before fitting, because growth curves deviate from logistic behaviour at later times under high CIP doses (see Discussion).

The key advantage of the simultaneous fit is that sharing a single parameter set $(r_0, K_0, q, c_0, \alpha)$ across all concentrations strongly constrains the posterior and prevents the identifiability issues that arise when fitting each curve independently near the MIC.

In [ ]:
# Load the full dose-response dataset (all media, replicates, time points)
DF = pd.read_csv('data/cip_dose_response_all.csv')

medium = 'standard'  # M9 + 0.2% glucose; alternatives: 'glucose', 'glycerol', 'TMP', 'CHL'
y0 = 0.001           # inoculum density OD_600 ~ 0.001 for the standard experiments

# Restrict to the standard medium and the first 12 h of each experiment
DF_CIP = DF[(DF.medium == medium) & (DF.Hours < 12)]

# Fit the model for each of the five biological replicates (separate days)
for rep in range(1, 6):
    DF_CIPrep = DF_CIP[DF_CIP.rep == rep]
    filename  = f'results/cip_model_{medium}_{rep}_12h.nc'
    fit_model(DF_CIPrep, filename, y0)

---
## 2. Independent logistic fits (comparison baseline)
*(Per-curve posteriors used in Figs. 1B, S2–S5 panel B)*

To illustrate the advantage of the simultaneous mechanistic fit, we also fit a plain logistic $\dot{N} = rN(1 - N/K)$ **independently** to each (concentration, replicate) combination.
Each fit has its own $(r, K)$ pair with no information shared across concentrations.

As shown in Fig. 1C–D, the independent fits become unreliable near and above the MIC: nearly flat growth curves are unidentifiable — they can be explained with a very large $K$ and tiny $r$, or vice versa. The mechanistic model avoids this degeneracy through the analytical constraints linking $r(c)$ and $K(c)$.

In [ ]:
# Fit a standard logistic to each (replicate, concentration) pair independently.
# Each fit is saved to its own NetCDF file for downstream plotting.

for rep in range(1, 6):
    DF_CIPrep    = DF_CIP[DF_CIP.rep == rep]
    unique_doses = sorted(DF_CIPrep['dosisAb'].unique())

    for c in unique_doses:
        DF2 = DF_CIPrep[DF_CIPrep.dosisAb == c]

        with pm.Model() as generative_model:
            t  = pm.Data("t",  DF2.Hours)
            y0 = pm.Data("y0", 0.001)

            # Uninformative uniform priors: no mechanistic link between concentrations
            K     = pm.Uniform('K', lower=0.0, upper=1.0)  # carrying capacity (OD)
            r     = pm.Uniform('r', lower=0.0, upper=2.0)  # growth rate (h^-1)
            sigma = pm.HalfNormal('sigma')

            likelihood = pm.Normal(
                "OD",
                mu=logistic(t, y0, r, K),
                sigma=sigma,
                observed=DF2.ODsmooth
            )

            idata = pm.sample(
                draws=1000, tune=2000, target_accept=0.95,
                cores=4, return_inferencedata=True
            )

            nc_path = f"results/cip_logistic_{medium}_{rep}_dose_{c}_12h.nc"
            idata.to_netcdf(nc_path, groups=["posterior", "sample_stats"])

---
## 3. Extract r(c) and K(c) from posteriors
*(CSV tables used in Figs. 1C–D, S2–S5 panels C–D)*

Given the fitted posterior for $(r_0, K_0, q, c_0, \alpha)$, we evaluate the analytical expressions (manuscript Eq. 4)

$$r(c) = \frac{r_0}{1+(c/c_0)^\alpha} - qc \qquad K(c) = K_0\left(1 - \frac{qc}{r_0}\left(1+\left(\frac{c}{c_0}\right)^\alpha\right)\right)$$

at each CIP concentration for 1 000 randomly drawn posterior samples, then summarise as mean ± SD.  The same extraction is done for the independent logistic posteriors, where $r$ and $K$ are direct MCMC parameters.

The resulting tables (`rK_{rep}_12h.csv`) feed the figure notebooks for panels C and D.

In [ ]:
# Unique CIP concentrations (shared across replicates)
unique_doses = sorted(DF_CIP['dosisAb'].unique())


def getKr(rep):
    """
    Posterior mean ± SD of r(c) and K(c) from the simultaneous mechanistic fit.
    Returns four lists (one value per CIP concentration): Ravg, Rstd, Kavg, Kstd.
    """
    samples   = az.from_netcdf(f"results/cip_model_standard_{rep}_12h.nc")
    posterior = samples.posterior.stack(samples=("draw", "chain"))

    Ravg, Kavg, Rstd, Kstd = [], [], [], []

    for c in unique_doses:
        Ri, Ki = [], []

        # Draw 1000 random samples from the posterior (Monte Carlo integration)
        for i in np.random.randint(0, posterior.samples.size, 1000):
            r0i    = posterior["r0"].values[i]
            K0i    = posterior["K0"].values[i]
            qi     = posterior["q"].values[i]
            c0i    = posterior["c0"].values[i]
            alphai = posterior["alpha"].values[i]

            # Evaluate K(c) and r(c) for this posterior sample (Eq. 4)
            ki = K0i - K0i * qi * c * (1 + (c / c0i) ** alphai) / r0i
            ri = r0i / (1 + (c / c0i) ** alphai) - qi * c

            # Above the MIC, K(c) and r(c) turn negative (population driven
            # to extinction). Clamp to zero so summary statistics are physical.
            if ki < 0:
                ki = 0
                ri = 0

            Ki.append(ki)
            Ri.append(ri)

        Ravg.append(np.average(Ri))
        Kavg.append(np.average(Ki))
        Rstd.append(np.std(Ri))
        Kstd.append(np.std(Ki))

    return Ravg, Rstd, Kavg, Kstd


def getKr_logistic(rep):
    """
    Posterior mean ± SD of r and K from the per-curve independent logistic fits.
    Here r and K are direct MCMC parameters, not derived quantities.
    """
    Ravg, Kavg, Rstd, Kstd = [], [], [], []

    for c in unique_doses:
        samples   = az.from_netcdf(f"results/cip_logistic_standard_{rep}_dose_{c}_12h.nc")
        posterior = samples.posterior.stack(samples=("draw", "chain"))

        Ri, Ki = [], []
        for i in np.random.randint(0, posterior.samples.size, 1000):
            Ki.append(posterior["K"].values[i])
            Ri.append(posterior["r"].values[i])

        Ravg.append(np.average(Ri))
        Kavg.append(np.average(Ki))
        Rstd.append(np.std(Ri))
        Kstd.append(np.std(Ki))

    return Ravg, Rstd, Kavg, Kstd


# Compute and save the r/K summary table for each replicate
for rep in range(1, 6):
    Ravg,  Rstd,  Kavg,  Kstd  = getKr(rep)
    RavgL, RstdL, KavgL, KstdL = getKr_logistic(rep)

    rK = pd.DataFrame({
        'c':     unique_doses,
        'Ravg':  Ravg,  'Rstd':  Rstd,  'Kavg':  Kavg,  'Kstd':  Kstd,
        'RavgL': RavgL, 'RstdL': RstdL, 'KavgL': KavgL, 'KstdL': KstdL
    })
    rK.to_csv(f'results/rK_{rep}_12h.csv', index=False)

---
## 4. MIC with fewer concentrations
*(Data used in Fig. 2A)*

A key practical question is: **how many CIP concentrations are needed for a reliable MIC estimate?**
To answer this, we refit the model using only the $n$ smallest concentrations (n = 3, 4, ..., 11), always including the zero-CIP control.
For each subset we compute the full posterior MIC distribution and record the absolute error relative to the MIC from the complete 11-concentration fit.

As shown in Fig. 2A, four concentrations are sufficient for an absolute error < 0.001 μg/mL (~8%), and five concentrations reduce it to < 10⁻⁵ μg/mL (< 0.1%).

In [ ]:
# Use replicate 2 as the reference (well-converged, representative)
rep = 2
filename  = f"results/cip_model_standard_{rep}_12h.nc"
idata_rep = az.from_netcdf(filename)

# Reference MIC from the complete 11-concentration fit
MICavg, MICstd = get_mic(idata_rep, output='simple')
print(f"Reference MIC (all concentrations): {MICavg:.4f} +/- {MICstd:.4f} ug/mL")

DF        = pd.read_csv('data/cip_dose_response_all.csv')
DF_CIP    = DF[(DF.medium == 'standard') & (DF.Hours < 12)]
DF_CIPrep = DF_CIP[DF_CIP.rep == rep]
CIP_conc  = sorted(DF_CIPrep.dosisAb.unique())  # ascending: 0 to 0.03 ug/mL

mic_convergence_results = []

# Incrementally add the next-lowest concentration and refit
for num_conc in range(3, len(CIP_conc) + 1):
    subset_conc = CIP_conc[:num_conc]   # keep the n smallest concentrations
    DF_subset   = DF_CIPrep[DF_CIPrep.dosisAb.isin(subset_conc)]
    print(f"\nFitting with {num_conc} concentrations: {subset_conc}")

    nc_filename = f"results/mic_calculations_add_{num_conc}_12h.nc"

    # To rerun fits, uncomment the next line and comment out the az.from_netcdf line:
    y0=0.001
    idata_subset = fit_model(DF_subset, nc_filename, y0)
    # idata_subset = az.from_netcdf(nc_filename)

    # Full posterior MIC distribution (needed to draw the box plots in Fig. 2A)
    mic_subset = get_mic(idata_subset, output='full')
    mic_convergence_results.append((num_conc, mic_subset))

# Flatten into a tidy DataFrame: one row per posterior sample per n
df = pd.DataFrame(mic_convergence_results, columns=['num_conc', 'mic_value'])
df = df.explode('mic_value')   # expand list column into individual rows
df.to_csv('results/mic_convergence_data_linear_add.csv', index=False)

---
## 5. MIC robustness — coefficient of variation
*(Data used in Fig. 2B)*

Even though individual parameters (especially $q$ and $c_0$) can have wide posteriors, the MIC is a tightly constrained derived quantity.
To quantify this, we compute the **coefficient of variation** (CV = SD / mean × 100 %) for each parameter and for the MIC, evaluated *across the posterior of each replicate* and then displayed together.

The sensitivity analysis in the manuscript (Eq. 9) shows analytically that the sensitivity of $c^*$ to $q$ is suppressed by a factor of $1 + \alpha\, x/(1+x)$ where $x = (c^*/c_0)^\alpha$.  Because $\alpha$ is large, the MIC CV stays low even when the CV of $q$ is high.

In [ ]:
def mic_cv(filename):
    """
    Load a fitted posterior and return the CV (%) for the MIC and each parameter.
    CV = posterior SD / posterior mean x 100.
    """
    idata_rep = az.from_netcdf(filename)

    MICavg, MICstd = get_mic(idata_rep)

    # Posterior mean and SD for each of the five model parameters
    r0avg    = idata_rep.posterior['r0'].mean()
    r0std    = idata_rep.posterior['r0'].std()
    qavg     = idata_rep.posterior['q'].mean()
    qstd     = idata_rep.posterior['q'].std()
    c0avg    = idata_rep.posterior['c0'].mean()
    c0std    = idata_rep.posterior['c0'].std()
    alphaavg = idata_rep.posterior['alpha'].mean()
    alphastd = idata_rep.posterior['alpha'].std()

    # Convert to percentage CV
    CVMIC   = MICstd   / MICavg   * 100
    CVr0    = r0std    / r0avg    * 100
    CVq     = qstd     / qavg     * 100
    CVc0    = c0std    / c0avg    * 100
    CValpha = alphastd / alphaavg * 100

    return CVMIC, float(CVr0), float(CVq), float(CVc0), float(CValpha)


L = []
for rep in range(1, 6):
    L.append(mic_cv(f"results/cip_model_standard_{rep}_12h.nc"))

DFL = pd.DataFrame.from_records(L, columns=['MIC', 'r0', 'q', 'c0', 'alpha'])
DFL.to_csv('results/cip_model_cv.csv', index=False)
print(DFL.round(1))

---
## 6. New medium conditions
*(Posteriors and parameter table used in Figs. 3–4)*

To test the generality of the model, we fit it to four medium perturbations at three doses each:

| Medium | Doses tested | Perturbation type |
|--------|-------------|-------------------|
| Glucose | 0.2%, 0.025%, 0.00625% (w/v) | Carbon-source variation |
| Glycerol | same three concentrations | Alternative carbon source |
| TMP | 0, 0.4, 1.0 µg/mL | Bacteriostatic (folate-synthesis inhibitor) |
| CHL | 0, 1.4, 6.0 µg/mL | Bacteriostatic (translation inhibitor) |

Each condition is fitted independently with its own five-parameter set, allowing us to track how the parameters shift (Fig. 3 lower panels) and to compare the resulting $r_0$–$q$ and $r_0$–MIC relationships (Fig. 4).

> **Note:** The inoculum for these experiments was $N_0 \approx 0.003$ (slightly higher than the standard 0.001), so `y0=0.003` is used here.

In [ ]:
DF = pd.read_csv('data/cip_dose_response_all.csv')

Medium = ['glucose', 'glycerol', 'TMP', 'CHL']
y0     = 0.003  # inoculum OD for these experiments (higher than the standard condition)

for medium in Medium:
    # Filter to this medium and the 12-hour observation window
    DFcond = DF[(DF.Hours < 12) & (DF.medium == medium)]
    COND   = sorted(DFcond.dosis.unique(), reverse=True)  # co-treatment/nutrient doses

    for cond in COND:
        DF2      = DFcond[DFcond.dosis == cond]
        filename = f'results/cip_model_{medium}_{cond}_12h.nc'
        print(f"\nFitting: {medium} at {cond}")
        # draws=4000 for richer posteriors in these single-replicate experiments
        fit_model(DF2, filename, y0, draws=4000)

### 6b. Assemble parameter table

After fitting, we collect the posterior mean and SD for all five parameters plus the MIC into a single CSV (`parameters_table.csv`).
This table is the source for Table 1 in the manuscript and feeds directly into the parameter subpanels of Fig. 3 and the cross-condition scatter plots of Fig. 4.

In [ ]:
def getPar(par, idata):
    """Return posterior mean and SD for a named model parameter."""
    Paravg = idata.posterior[par].mean()
    Parstd = idata.posterior[par].std()
    return Paravg, Parstd


DF        = pd.read_csv('data/cip_dose_response_all.csv')
Medium    = ['glucose', 'glycerol', 'TMP', 'CHL']
rows_list = []

for medium in Medium:
    DFcond = DF[(DF.Hours < 12) & (DF.medium == medium)]
    COND   = sorted(DFcond.dosis.unique(), reverse=True)

    for cond in COND:
        filename = f'results/cip_model_{medium}_{cond}_12h.nc'
        idata    = az.from_netcdf(filename)

        current_row = {'medium': medium, 'concentration': cond}

        for par in ["r0", "K0", "q", "c0", "alpha", "MIC"]:
            # MIC is a derived quantity; all others are direct posterior parameters
            if par == "MIC":
                avg, std = get_mic(idata)
            else:
                avg, std = getPar(par, idata)

            current_row[f'{par}_mean'] = float(avg)
            current_row[f'{par}_std']  = float(std)

        rows_list.append(current_row)

df_final = pd.DataFrame(rows_list)
df_final.set_index('medium', inplace=True)
df_final.to_csv('results/parameters_table.csv')
print(df_final.round(4))

---
## 7. Fixed-α fits (sensitivity analysis)
*(Posteriors and parameter table used in Fig. S6)*

The Hill coefficient $\alpha$ is the most stable parameter across conditions (range 3.1–7.7, mean 6.2; Table 1).
A natural question is whether fixing $\alpha$ at its global mean simplifies the posterior and reduces uncertainty in $q$ and $c_0$, while preserving the main biological findings.

We repeat the fits from Section 6 using `fit_model_fixed_alpha()`, which constrains $\alpha \sim \mathcal{N}(6.21, 0.01^2)$ — effectively a fixed value while remaining technically differentiable for NUTS.

In [ ]:
DF = pd.read_csv('data/cip_dose_response_all.csv')

Medium = ['glucose', 'glycerol', 'TMP', 'CHL']
y0     = 0.003

for medium in Medium:
    DFcond = DF[(DF.Hours < 12) & (DF.medium == medium)]
    COND   = sorted(DFcond.dosis.unique(), reverse=True)

    for cond in COND:
        DF2      = DFcond[DFcond.dosis == cond]
        filename = f'results/cip_model_{medium}_{cond}_12h_fixed_alpha.nc'
        print(f"\nFitting (fixed alpha): {medium} at {cond}")
        fit_model_fixed_alpha(DF2, filename, y0)

In [ ]:
# Assemble the parameter table for fixed-alpha fits (same structure as Section 6b)

def getPar(par, idata):
    Paravg = idata.posterior[par].mean()
    Parstd = idata.posterior[par].std()
    return Paravg, Parstd


DF        = pd.read_csv('data/cip_dose_response_all.csv')
Medium    = ['glucose', 'glycerol', 'TMP', 'CHL']
rows_list = []

for medium in Medium:
    DFcond = DF[(DF.Hours < 12) & (DF.medium == medium)]
    COND   = sorted(DFcond.dosis.unique(), reverse=True)

    for cond in COND:
        filename = f'results/cip_model_{medium}_{cond}_12h_fixed_alpha.nc'
        idata    = az.from_netcdf(filename)

        current_row = {'medium': medium, 'concentration': cond}

        for par in ["r0", "K0", "q", "c0", "alpha", "MIC"]:
            avg, std = get_mic(idata) if par == "MIC" else getPar(par, idata)
            current_row[f'{par}_mean'] = float(avg)
            current_row[f'{par}_std']  = float(std)

        rows_list.append(current_row)

df_final = pd.DataFrame(rows_list)
df_final.set_index('medium', inplace=True)
df_final.to_csv('results/parameters_table_fixed_alpha.csv')
print(df_final.round(4))